# 03 — Modelagem

Estimação dos modelos de Poisson e Binomial Negativa com offset de exposição.

**Offset:** log(n_reviews) — controla pela exposição de cada vendedor, modelando a **taxa** de negativos.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import json, warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/dataset_vendedores.csv")
print(f"Dataset: {len(df):,} vendedores")

Dataset: 1,754 vendedores


## 1. Preparação das variáveis

In [2]:
# Agrupar categorias com menos de 30 vendedores
cat_counts = df["categoria_principal"].value_counts()
cats_keep = cat_counts[cat_counts >= 30].index.tolist()
df["cat_group"] = df["categoria_principal"].where(df["categoria_principal"].isin(cats_keep), "other")
print(f"Categorias: {df['categoria_principal'].nunique()} -> {df['cat_group'].nunique()} agrupadas")
print(df["cat_group"].value_counts())

Categorias: 65 -> 19 agrupadas
cat_group
other                       318
sports_leisure              188
health_beauty               169
housewares                  163
furniture_decor             117
auto                        106
computers_accessories        96
bed_bath_table               85
toys                         79
baby                         56
garden_tools                 51
telephony                    48
pet_shop                     48
perfumery                    47
cool_stuff                   44
stationery                   39
watches_gifts                35
fashion_bags_accessories     35
electronics                  30
Name: count, dtype: int64


In [3]:
# Agrupar estados com menos de 30 vendedores
st_counts = df["seller_state"].value_counts()
st_keep = st_counts[st_counts >= 30].index.tolist()
df["state_group"] = df["seller_state"].where(df["seller_state"].isin(st_keep), "other")
print(f"Estados: {df['seller_state'].nunique()} -> {df['state_group'].nunique()} agrupados")
print(df["state_group"].value_counts())

Estados: 20 -> 7 agrupados
state_group
SP       1076
PR        190
MG        145
RJ         98
SC         95
other      92
RS         58
Name: count, dtype: int64


In [4]:
# Offset e dummies
df["log_n_reviews"] = np.log(df["n_reviews"])

df_model = df[["n_negative", "log_n_reviews", "atraso_medio", "ticket_medio",
               "frete_medio", "peso_medio", "cat_group", "state_group"]].copy()
df_model = pd.get_dummies(df_model, columns=["cat_group", "state_group"], drop_first=True, dtype=int)

y = df_model["n_negative"]
offset = df_model["log_n_reviews"]
X = df_model.drop(columns=["n_negative", "log_n_reviews"])
X = sm.add_constant(X)

print(f"Y: {y.shape}, X: {X.shape}")

Y: (1754,), X: (1754, 29)


## 2. Modelo de Poisson

In [5]:
poisson = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset).fit()
print(poisson.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             n_negative   No. Observations:                 1754
Model:                            GLM   Df Residuals:                     1725
Model Family:                 Poisson   Df Model:                           28
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3715.3
Date:                Thu, 16 Apr 2026   Deviance:                       3002.8
Time:                        00:36:32   Pearson chi2:                 3.09e+03
No. Iterations:                     6   Pseudo R-squ. (CS):             0.2597
Covariance Type:            nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
cons

In [6]:
print(f"Log-Likelihood: {poisson.llf:.2f}")
print(f"AIC: {poisson.aic:.2f}")
print(f"BIC: {poisson.bic_llf:.2f}")
print(f"Deviance: {poisson.deviance:.2f}")
print(f"Pearson chi2/df: {poisson.pearson_chi2/poisson.df_resid:.4f}  (>>1 indica sobredispersao)")

Log-Likelihood: -3715.34
AIC: 7488.67
BIC: 7647.29
Deviance: 3002.79
Pearson chi2/df: 1.7887  (>>1 indica sobredispersao)


## 3. Teste de sobredispersão (Cameron-Trivedi)

- H0: Equidispersão (Poisson adequado)
- H1: Sobredispersão (BN necessária)

In [7]:
mu_hat = poisson.fittedvalues
aux_dep = ((y - mu_hat)**2 - y) / mu_hat
aux_X = sm.add_constant(mu_hat)
aux_ols = sm.OLS(aux_dep, aux_X).fit()

alpha_est = aux_ols.params.iloc[1]
t_stat = aux_ols.tvalues.iloc[1]
p_val_ct = aux_ols.pvalues.iloc[1]

print("=== Teste de Cameron-Trivedi ===")
print(f"Alpha estimado:  {alpha_est:.6f}")
print(f"Estatistica t:   {t_stat:.4f}")
print(f"p-valor:         {p_val_ct:.2e}")
if p_val_ct < 0.05 and alpha_est > 0:
    print("\nRejeita H0: sobredispersao confirmada. BN necessaria.")
else:
    print("\nNao rejeita H0: Poisson pode ser adequado.")

=== Teste de Cameron-Trivedi ===
Alpha estimado:  0.044898
Estatistica t:   6.2948
p-valor:         3.88e-10

Rejeita H0: sobredispersao confirmada. BN necessaria.


## 4. Modelo Binomial Negativo

In [8]:
alpha_mm = max((poisson.pearson_chi2 / poisson.df_resid - 1) / mu_hat.mean(), 0.01)
print(f"Alpha (metodo dos momentos): {alpha_mm:.4f}")

negbin = sm.GLM(y, X, family=sm.families.NegativeBinomial(alpha=alpha_mm), offset=offset).fit()
print(negbin.summary())

Alpha (metodo dos momentos): 0.1147
                 Generalized Linear Model Regression Results                  
Dep. Variable:             n_negative   No. Observations:                 1754
Model:                            GLM   Df Residuals:                     1725
Model Family:        NegativeBinomial   Df Model:                           28
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3469.3
Date:                Thu, 16 Apr 2026   Deviance:                       1831.6
Time:                        00:36:32   Pearson chi2:                 1.80e+03
No. Iterations:                     8   Pseudo R-squ. (CS):             0.1200
Covariance Type:            nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------

In [9]:
print(f"Log-Likelihood: {negbin.llf:.2f}")
print(f"AIC: {negbin.aic:.2f}")
print(f"BIC: {negbin.bic_llf:.2f}")
print(f"Deviance: {negbin.deviance:.2f}")
print(f"Pearson chi2/df: {negbin.pearson_chi2/negbin.df_resid:.4f}")

Log-Likelihood: -3469.33
AIC: 6996.66
BIC: 7155.28
Deviance: 1831.60
Pearson chi2/df: 1.0452


## 5. Comparação dos modelos

In [10]:
lr_stat = 2 * (negbin.llf - poisson.llf)
p_value_lr = stats.chi2.sf(abs(lr_stat), df=1)

print("=== Teste LR: Poisson vs BN ===")
print(f"LR stat: {abs(lr_stat):.2f}")
print(f"p-valor: {p_value_lr:.2e}")

fmt = "{:<20s} {:>15s} {:>15s} {:>10s}"
print("\n" + fmt.format("Metrica", "Poisson", "NegBin", "Melhor"))
print("-" * 63)
for name, vp, vn, d in [
    ("Log-Likelihood", poisson.llf, negbin.llf, "maior"),
    ("AIC", poisson.aic, negbin.aic, "menor"),
    ("BIC", poisson.bic_llf, negbin.bic_llf, "menor"),
    ("Deviance", poisson.deviance, negbin.deviance, "menor"),
    ("Pearson chi2/df", poisson.pearson_chi2/poisson.df_resid,
     negbin.pearson_chi2/negbin.df_resid, "proximo_1"),
]:
    if d == "maior": m = "NegBin" if vn > vp else "Poisson"
    elif d == "menor": m = "NegBin" if vn < vp else "Poisson"
    else: m = "NegBin" if abs(vn-1) < abs(vp-1) else "Poisson"
    print(f"{name:<20s} {vp:>15.2f} {vn:>15.2f} {m:>10s}")

=== Teste LR: Poisson vs BN ===
LR stat: 492.01
p-valor: 5.21e-109

Metrica                      Poisson          NegBin     Melhor
---------------------------------------------------------------
Log-Likelihood              -3715.34        -3469.33     NegBin
AIC                          7488.67         6996.66     NegBin
BIC                          7647.29         7155.28     NegBin
Deviance                     3002.79         1831.60     NegBin
Pearson chi2/df                 1.79            1.05     NegBin


## 6. Coeficientes e significância

In [11]:
coefs = pd.DataFrame({
    "coef": negbin.params, "std_err": negbin.bse,
    "z": negbin.tvalues, "p_value": negbin.pvalues,
    "exp_coef": np.exp(negbin.params)
})
coefs["sig"] = coefs["p_value"].apply(
    lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
)

sig = coefs[coefs["p_value"] < 0.05]
nsig = coefs[coefs["p_value"] >= 0.05]

print(f"Significativas: {len(sig)}")
print(sig[["coef", "exp_coef", "p_value", "sig"]].to_string())
print(f"\nNao significativas: {len(nsig)}")
if len(nsig) > 0:
    print(nsig[["coef", "exp_coef", "p_value"]].to_string())

Significativas: 6
                               coef  exp_coef       p_value  sig
const                     -1.824545  0.161291  5.703199e-63  ***
atraso_medio               0.040666  1.041504  7.221695e-19  ***
frete_medio                0.003348  1.003354  3.163292e-02    *
cat_group_bed_bath_table   0.242405  1.274310  7.068260e-03   **
cat_group_furniture_decor  0.217027  1.242378  1.584514e-02    *
cat_group_telephony        0.287219  1.332716  7.614477e-03   **

Nao significativas: 23
                                        coef  exp_coef   p_value
ticket_medio                        0.000117  1.000117  0.227008
peso_medio                          0.000012  1.000012  0.103854
cat_group_baby                     -0.065390  0.936702  0.557321
cat_group_computers_accessories     0.174390  1.190520  0.059124
cat_group_cool_stuff               -0.167049  0.846158  0.178038
cat_group_electronics              -0.031422  0.969066  0.818430
cat_group_fashion_bags_accessories -0.195704  0.

## 7. Modelo reduzido

In [12]:
vars_remover = [v for v in nsig.index.tolist() if v != "const"]

if len(vars_remover) > 0:
    print(f"Removendo {len(vars_remover)} variaveis nao significativas:")
    for v in vars_remover:
        print(f"  - {v} (p={coefs.loc[v, 'p_value']:.4f})")
    X_red = X.drop(columns=vars_remover)
    negbin_red = sm.GLM(y, X_red, family=sm.families.NegativeBinomial(alpha=alpha_mm), offset=offset).fit()
    print(f"\nModelo Reduzido: {X_red.shape[1]} vars (vs {X.shape[1]})")
    print(f"AIC: {negbin_red.aic:.2f} (vs {negbin.aic:.2f})")
    print(f"BIC: {negbin_red.bic_llf:.2f} (vs {negbin.bic_llf:.2f})")
    print(negbin_red.summary())
else:
    print("Todas significativas. Modelo reduzido = completo.")
    negbin_red = negbin
    X_red = X

Removendo 23 variaveis nao significativas:
  - ticket_medio (p=0.2270)
  - peso_medio (p=0.1039)
  - cat_group_baby (p=0.5573)
  - cat_group_computers_accessories (p=0.0591)
  - cat_group_cool_stuff (p=0.1780)
  - cat_group_electronics (p=0.8184)
  - cat_group_fashion_bags_accessories (p=0.1628)
  - cat_group_garden_tools (p=0.7106)
  - cat_group_health_beauty (p=0.6839)
  - cat_group_housewares (p=0.6821)
  - cat_group_other (p=0.4985)
  - cat_group_perfumery (p=0.6002)
  - cat_group_pet_shop (p=0.3157)
  - cat_group_sports_leisure (p=0.6602)
  - cat_group_stationery (p=0.2346)
  - cat_group_toys (p=0.3797)
  - cat_group_watches_gifts (p=0.2612)
  - state_group_PR (p=0.5726)
  - state_group_RJ (p=0.3137)
  - state_group_RS (p=0.3266)
  - state_group_SC (p=0.4083)
  - state_group_SP (p=0.2026)
  - state_group_other (p=0.8618)

Modelo Reduzido: 6 vars (vs 29)
AIC: 6982.97 (vs 6996.66)
BIC: 7015.78 (vs 7155.28)
                 Generalized Linear Model Regression Results                 

## 8. Salvar resultados

In [13]:
coefs_final = pd.DataFrame({
    "coef": negbin_red.params, "std_err": negbin_red.bse,
    "z": negbin_red.tvalues, "p_value": negbin_red.pvalues,
    "exp_coef": np.exp(negbin_red.params)
})
coefs_final.to_csv("../outputs/coeficientes_negbin.csv")

metricas = {
    "poisson": {"aic": float(poisson.aic), "bic": float(poisson.bic_llf),
        "llf": float(poisson.llf), "deviance": float(poisson.deviance),
        "pearson_chi2_df": float(poisson.pearson_chi2/poisson.df_resid), "n_vars": int(X.shape[1])},
    "negbin_completo": {"aic": float(negbin.aic), "bic": float(negbin.bic_llf),
        "llf": float(negbin.llf), "deviance": float(negbin.deviance),
        "pearson_chi2_df": float(negbin.pearson_chi2/negbin.df_resid),
        "n_vars": int(X.shape[1]), "alpha": float(alpha_mm)},
    "negbin_reduzido": {"aic": float(negbin_red.aic), "bic": float(negbin_red.bic_llf),
        "llf": float(negbin_red.llf), "deviance": float(negbin_red.deviance),
        "pearson_chi2_df": float(negbin_red.pearson_chi2/negbin_red.df_resid),
        "n_vars": int(X_red.shape[1]), "alpha": float(alpha_mm)},
    "cameron_trivedi": {"alpha": float(alpha_est), "t": float(t_stat), "p_value": float(p_val_ct)},
    "lr_test": {"statistic": float(abs(lr_stat)), "p_value": float(p_value_lr)}
}
with open("../outputs/metricas_modelos.json", "w") as f:
    json.dump(metricas, f, indent=2)
print("Salvos: coeficientes_negbin.csv e metricas_modelos.json")

Salvos: coeficientes_negbin.csv e metricas_modelos.json


## 9. Performance do modelo (taxa observada vs prevista)

Visual de aderência entre taxa observada e taxa prevista pelo modelo Binomial Negativo reduzido.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Predicoes em contagem e conversao para taxa (%) por review
pred_count = negbin_red.fittedvalues
obs_rate = (y / np.exp(offset)) * 100
pred_rate = (pred_count / np.exp(offset)) * 100

rmse_rate = np.sqrt(np.mean((obs_rate - pred_rate) ** 2))
mae_rate = np.mean(np.abs(obs_rate - pred_rate))
corr_rate = np.corrcoef(obs_rate, pred_rate)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter observado vs previsto
axes[0].scatter(pred_rate, obs_rate, alpha=0.45, s=25, color="#1f77b4")
line_min = min(pred_rate.min(), obs_rate.min())
line_max = max(pred_rate.max(), obs_rate.max())
axes[0].plot([line_min, line_max], [line_min, line_max], "r--", lw=2, label="Linha ideal")
axes[0].set_title("Taxa observada vs prevista")
axes[0].set_xlabel("Taxa prevista de negativos (%)")
axes[0].set_ylabel("Taxa observada de negativos (%)")
axes[0].legend()

# Calibracao por decis de risco previsto
df_perf = pd.DataFrame({"obs_rate": obs_rate, "pred_rate": pred_rate}).sort_values("pred_rate")
df_perf["decil"] = pd.qcut(df_perf["pred_rate"], 10, labels=False, duplicates="drop")
calib = df_perf.groupby("decil", observed=True).agg(
    obs_media=("obs_rate", "mean"),
    pred_media=("pred_rate", "mean")
).reset_index()

axes[1].plot(calib["decil"] + 1, calib["obs_media"], marker="o", label="Observada")
axes[1].plot(calib["decil"] + 1, calib["pred_media"], marker="s", label="Prevista")
axes[1].set_title("Calibracao por decis")
axes[1].set_xlabel("Decil de risco previsto")
axes[1].set_ylabel("Taxa media de negativos (%)")
axes[1].legend()

plt.suptitle(
    f"Performance do modelo | RMSE={rmse_rate:.2f} p.p. | MAE={mae_rate:.2f} p.p. | Corr={corr_rate:.2f}",
    fontsize=12
)
plt.tight_layout()
plt.savefig("../outputs/modelo_performance_taxa.png", bbox_inches="tight")
plt.show()

print("Grafico salvo: ../outputs/modelo_performance_taxa.png")

## 10. Graficos animados (GIF) para LinkedIn

Versoes animadas dos principais achados para aumentar engajamento no post.

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

os.makedirs("../outputs/animations", exist_ok=True)

# Base para interpretacao visual
pred_count = negbin_red.fittedvalues
obs_rate = (y / np.exp(offset)) * 100
pred_rate = (pred_count / np.exp(offset)) * 100

coefs_plot = pd.DataFrame({
    "var": negbin_red.params.index,
    "coef": negbin_red.params.values,
    "p": negbin_red.pvalues.values
})
coefs_plot = coefs_plot[(coefs_plot["var"] != "const") & (coefs_plot["p"] < 0.05)].copy()
coefs_plot["irr"] = np.exp(coefs_plot["coef"])
coefs_plot["impacto_pct"] = (coefs_plot["irr"] - 1) * 100
coefs_plot = coefs_plot.sort_values("impacto_pct")

# 1) GIF: impacto percentual dos coeficientes significativos
fig, ax = plt.subplots(figsize=(10, 5))
vals = np.zeros(len(coefs_plot))
bars = ax.barh(coefs_plot["var"], vals, color="#4C78A8")
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.set_xlim(min(coefs_plot["impacto_pct"].min() * 1.2, -5), max(coefs_plot["impacto_pct"].max() * 1.2, 5))
ax.set_xlabel("Impacto na taxa de insatisfacao (%)")
ax.set_title("Efeito estimado por variavel (IRR)")

labels = []
for b in bars:
    txt = ax.text(0, b.get_y() + b.get_height()/2, "", va="center", ha="left", fontsize=9)
    labels.append(txt)

def update_bars(frame):
    frac = (frame + 1) / 40
    for i, b in enumerate(bars):
        target = coefs_plot.iloc[i]["impacto_pct"]
        current = target * frac
        b.set_width(current)
        labels[i].set_position((current + (0.6 if current >= 0 else -0.6), b.get_y() + b.get_height()/2))
        labels[i].set_ha("left" if current >= 0 else "right")
        labels[i].set_text(f"{current:+.1f}%")
    return [*bars, *labels]

anim1 = FuncAnimation(fig, update_bars, frames=40, interval=70, blit=False)
anim1.save("../outputs/animations/anim_impacto_variaveis.gif", writer=PillowWriter(fps=15))
plt.close(fig)

# 2) GIF: observado vs previsto (pontos aparecendo progressivamente)
perf_df = pd.DataFrame({"pred": pred_rate, "obs": obs_rate}).sort_values("pred").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7, 7))
line_min = min(perf_df["pred"].min(), perf_df["obs"].min())
line_max = max(perf_df["pred"].max(), perf_df["obs"].max())
ax.plot([line_min, line_max], [line_min, line_max], "r--", lw=1.5, label="Linha ideal")
sc = ax.scatter([], [], s=22, alpha=0.45, color="#1F77B4")
ax.set_xlim(line_min, line_max)
ax.set_ylim(line_min, line_max)
ax.set_xlabel("Taxa prevista de negativos (%)")
ax.set_ylabel("Taxa observada de negativos (%)")
ax.set_title("Performance do modelo: observado vs previsto")
ax.legend(loc="upper left")

rmse_rate = np.sqrt(np.mean((obs_rate - pred_rate) ** 2))
mae_rate = np.mean(np.abs(obs_rate - pred_rate))
corr_rate = np.corrcoef(obs_rate, pred_rate)[0, 1]
metric_text = ax.text(
    0.03, 0.97,
    f"RMSE={rmse_rate:.2f} p.p.\nMAE={mae_rate:.2f} p.p.\nCorr={corr_rate:.2f}",
    transform=ax.transAxes,
    va="top",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none")
)

def update_scatter(frame):
    n = int((frame + 1) / 45 * len(perf_df))
    pts = perf_df.iloc[:n]
    sc.set_offsets(np.c_[pts["pred"], pts["obs"]])
    return sc, metric_text

anim2 = FuncAnimation(fig, update_scatter, frames=45, interval=70, blit=False)
anim2.save("../outputs/animations/anim_performance_modelo.gif", writer=PillowWriter(fps=15))
plt.close(fig)

# 3) GIF: calibracao por decis
calib_df = pd.DataFrame({"obs_rate": obs_rate, "pred_rate": pred_rate}).sort_values("pred_rate")
calib_df["decil"] = pd.qcut(calib_df["pred_rate"], 10, labels=False, duplicates="drop")
calib = calib_df.groupby("decil", observed=True).agg(
    obs_media=("obs_rate", "mean"),
    pred_media=("pred_rate", "mean")
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
x = calib["decil"] + 1
line_obs, = ax.plot([], [], marker="o", linewidth=2, label="Observada", color="#E45756")
line_pred, = ax.plot([], [], marker="s", linewidth=2, label="Prevista", color="#4C78A8")
ax.set_xlim(1, x.max())
ax.set_ylim(0, max(calib[["obs_media", "pred_media"]].max()) * 1.15)
ax.set_xlabel("Decil de risco previsto")
ax.set_ylabel("Taxa media de negativos (%)")
ax.set_title("Calibracao do modelo por faixa de risco")
ax.legend()

def update_calib(frame):
    n = frame + 1
    line_obs.set_data(x.iloc[:n], calib["obs_media"].iloc[:n])
    line_pred.set_data(x.iloc[:n], calib["pred_media"].iloc[:n])
    return line_obs, line_pred

anim3 = FuncAnimation(fig, update_calib, frames=len(calib), interval=250, blit=False)
anim3.save("../outputs/animations/anim_calibracao_decis.gif", writer=PillowWriter(fps=4))
plt.close(fig)

print("GIFs salvos em ../outputs/animations:")
print("- anim_impacto_variaveis.gif")
print("- anim_performance_modelo.gif")
print("- anim_calibracao_decis.gif")